# V19: OAV-Filtered CAS-Level Fingerprint vs Free Sorting

**Hypothesis:** The OT1-category approach (V18) loses ingredient identity — two recipes both labelled 'fruity' look identical even if they use different molecules. And sub-threshold ingredients add noise.

**This notebook tests:**
1. **CAS-level fingerprint** — one dimension per unique CAS number (137 dimensions vs ~12 OT1 categories)
2. **OAV-based weighting** — each ingredient weighted by `Totalmenge / threshold_ppm` (proxy for perceived intensity)
3. **OAV cutoff filtering** — zero out ingredients below X% of a recipe's total OAV (removes sub-threshold noise)

**Goal:** Improve on V18's 45.5% alignment with the panelist free sorting result.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from scipy.optimize import linear_sum_assignment
from sklearn.manifold import MDS
from sklearn.preprocessing import normalize
import openpyxl
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = '../outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLUSTER_COLORS = [
    '#E63946', '#457B9D', '#2A9D8F', '#E9C46A',
    '#F4A261', '#A8DADC', '#6A0572', '#264653',
]
print('Libraries loaded.')

## 1. Load & Preprocess Ingredient Data

In [ ]:
DATA_PATH   = '../data/gold/Versuchsdaten_3_1.csv'
IGNORE_PATH = '../data/gold/ignone_substances.csv'
CAS_PATH    = '../data/gold/CAS Nummern.csv'

df_raw = pd.read_csv(DATA_PATH)
ign    = pd.read_csv(IGNORE_PATH)
cas    = pd.read_csv(CAS_PATH, header=13)

ign_cas = ign[['Ident']].merge(
    cas[['Ident.', 'CAS-Nr.: - Hinweis 1']].rename(columns={'Ident.': 'Ident'}),
    on='Ident', how='left'
)
cas_to_ignore   = set(ign_cas['CAS-Nr.: - Hinweis 1'].dropna().astype(str).str.strip())
names_to_ignore = {str(n).lower().strip() for n in ign['Name']}

df = df_raw.copy()
df['_cas'] = df['CAS-Nr.: - Hinweis 1'].astype(str).str.strip()
df.loc[df['_cas'].isin(cas_to_ignore), 'Totalmenge'] = 0.0
df.drop(columns='_cas', inplace=True)
df.loc[df['Name'].str.lower().str.strip().isin(names_to_ignore), 'Totalmenge'] = 0.0

per_recipe_total = df.groupby('Rez.-Nr.')['Totalmenge'].transform('sum')
df['Totalmenge'] = np.where(per_recipe_total > 0,
                            df['Totalmenge'] / per_recipe_total,
                            df['Totalmenge'])

# Parse threshold — cap zero at 0.001 to avoid inf OAV
def parse_thresh(v, fallback=None):
    try:
        t = float(str(v).strip().replace(',', '.'))
        return max(t, 0.001) if t >= 0 else fallback
    except:
        return fallback

df['thresh_ppm'] = df['Threshold ppm (Datenbank)'].map(parse_thresh)

# OAV proxy: Totalmenge / threshold_ppm  (fallback to Totalmenge if no threshold)
df['oav_proxy'] = np.where(
    df['thresh_ppm'].notna() & (df['Totalmenge'] > 0),
    df['Totalmenge'] / df['thresh_ppm'],
    df['Totalmenge'],
)

recipes = df['Rez.-Nr.'].unique().tolist()
print(f'Recipes: {len(recipes)}')
print(f'Active ingredient rows: {(df["Totalmenge"] > 0).sum()}')
print(f'Threshold coverage: {df[df["Totalmenge"]>0]["thresh_ppm"].notna().mean()*100:.1f}%')
print(f'Unique CAS numbers: {df["CAS-Nr.: - Hinweis 1"].nunique()}')

## 2. Load Free Sorting Reference (same as V17/V18)

In [ ]:
EXCEL_PATH = Path('../free_sorting/Auswertung Best Of XLStat Erdbeere.xlsm')
wb = openpyxl.load_workbook(EXCEL_PATH, keep_vba=True, data_only=True)

ws_data = wb['Tabelle1 (7)']
rows_fs = list(ws_data.iter_rows(min_row=1, max_row=26, values_only=True))
fs_products = [r[0] for r in rows_fs[1:] if r[0] is not None]
fs_matrix   = np.array([[r[i] for i in range(1, 13)] for r in rows_fs[1:] if r[0] is not None], dtype=float)

n_fs = len(fs_products)
co_occ = np.zeros((n_fs, n_fs))
for a in range(12):
    g = fs_matrix[:, a]
    for i in range(n_fs):
        for j in range(n_fs):
            if g[i] == g[j]: co_occ[i, j] += 1
fs_diss = 1.0 - co_occ / 12
np.fill_diagonal(fs_diss, 0.0)

ws_xlstat = wb['Free Sorting7']
xlstat_rows = list(ws_xlstat.iter_rows(min_row=132, max_row=157, values_only=True))
xlstat_products, xlstat_d1, xlstat_d2 = [], [], []
for row in xlstat_rows:
    vals = [v for v in row if v is not None]
    if vals and isinstance(vals[0], str) and vals[0].strip() and isinstance(vals[1], (int, float)):
        xlstat_products.append(vals[0])
        xlstat_d1.append(float(vals[1]))
        xlstat_d2.append(float(vals[2]))
xlstat_coords = np.column_stack([xlstat_d1, xlstat_d2])

FS_THRESHOLD = 1.2
Z_fs = linkage(squareform(fs_diss, checks=False), method='ward')
labels_fs = fcluster(Z_fs, t=FS_THRESHOLD, criterion='distance')
labels_fs_xlstat = np.array([labels_fs[fs_products.index(p)] for p in xlstat_products])

print(f'Free sorting: {n_fs} products, {len(np.unique(labels_fs))} clusters at t={FS_THRESHOLD}')

## 3. Build CAS-Level OAV Fingerprint

**Key difference from V18:**
- V18: grouped by OT1 category (~12 dimensions)
- V19: one dimension per unique CAS number (137 dimensions)
- Each dimension = OAV proxy = `Totalmenge / threshold_ppm`

Then apply **OAV cutoff**: zero out any ingredient below X% of a recipe's total OAV.  
This removes ingredients the nose can't detect.

In [ ]:
def build_cas_vectors(df, recipes, oav_cutoff_pct=0.01):
    """
    Build CAS-level OAV fingerprint.
    
    oav_cutoff_pct: zero out ingredients contributing less than this fraction
                    of the recipe's total OAV (e.g. 0.01 = 1% cutoff).
                    Set to 0 to keep all ingredients.
    """
    # Build CAS vocabulary (active, non-ignored ingredients only)
    active = df[df['Totalmenge'] > 0].copy()
    cas_vocab = sorted(active['CAS-Nr.: - Hinweis 1'].astype(str).str.strip().unique())
    cas_to_idx = {c: i for i, c in enumerate(cas_vocab)}
    
    n = len(recipes)
    d = len(cas_vocab)
    vectors = np.zeros((n, d), dtype=np.float64)
    
    for r_idx, recipe in enumerate(recipes):
        rows = df[(df['Rez.-Nr.'] == recipe) & (df['Totalmenge'] > 0)]
        total_oav = rows['oav_proxy'].sum()
        
        for _, row in rows.iterrows():
            oav = row['oav_proxy']
            # Apply OAV cutoff
            if total_oav > 0 and oav / total_oav < oav_cutoff_pct:
                continue
            cas = str(row['CAS-Nr.: - Hinweis 1']).strip()
            if cas in cas_to_idx:
                vectors[r_idx, cas_to_idx[cas]] += oav
    
    # Count active ingredients per recipe after cutoff
    n_active = (vectors > 0).sum(axis=1)
    print(f'  CAS dimensions: {d}')
    print(f'  Active ingredients per recipe: min={n_active.min()}, '
          f'median={np.median(n_active):.0f}, max={n_active.max()}')
    print(f'  Zero vectors: {(vectors.sum(axis=1) == 0).sum()}')
    
    return cas_vocab, normalize(vectors)

def cosine_dissimilarity(vecs):
    sim = np.clip(vecs @ vecs.T, -1.0, 1.0)
    diss = 1.0 - sim
    np.fill_diagonal(diss, 0.0)
    return diss

def run_mds(diss, threshold):
    mds = MDS(n_components=2, dissimilarity='precomputed', metric=True,
              n_init=10, max_iter=1000, random_state=42, normalized_stress='auto')
    coords = mds.fit_transform(diss)
    Z = linkage(squareform(diss, checks=False), method='ward')
    labels = fcluster(Z, t=threshold, criterion='distance')
    return coords, Z, labels, mds.stress_

# Test with no cutoff first
print('No cutoff (all ingredients):')
_, vecs_0 = build_cas_vectors(df, recipes, oav_cutoff_pct=0.0)
print()
print('1% OAV cutoff:')
_, vecs_1 = build_cas_vectors(df, recipes, oav_cutoff_pct=0.01)
print()
print('5% OAV cutoff:')
_, vecs_5 = build_cas_vectors(df, recipes, oav_cutoff_pct=0.05)
print()
print('10% OAV cutoff:')
_, vecs_10 = build_cas_vectors(df, recipes, oav_cutoff_pct=0.10)

## 4. Plot Helpers

In [ ]:
def confidence_ellipse(x, y, ax, n_std=1.5, **kwargs):
    if len(x) < 2:
        ax.scatter(x, y, s=80, color=kwargs.get('facecolor', 'gray'), zorder=4)
        return
    cov = np.cov(x, y)
    ev, evec = np.linalg.eigh(cov)
    order = ev.argsort()[::-1]
    ev, evec = ev[order], evec[:, order]
    angle  = np.degrees(np.arctan2(*evec[:, 0][::-1]))
    width  = max(2 * n_std * np.sqrt(abs(ev[0])), 0.001)
    height = max(2 * n_std * np.sqrt(abs(ev[1])), 0.001)
    ax.add_patch(Ellipse(xy=(np.mean(x), np.mean(y)),
                         width=width, height=height, angle=angle, **kwargs))

def mds_plot(ax, coords, names, cluster_labels, title):
    colors = CLUSTER_COLORS[:len(np.unique(cluster_labels))]
    ax.set_facecolor('#FAFAFA')
    ax.axhline(0, color='#CCCCCC', lw=0.7, zorder=1)
    ax.axvline(0, color='#CCCCCC', lw=0.7, zorder=1)
    for c_idx, cid in enumerate(sorted(np.unique(cluster_labels))):
        mask = cluster_labels == cid
        cx, cy = coords[mask, 0], coords[mask, 1]
        col = colors[c_idx % len(colors)]
        confidence_ellipse(cx, cy, ax, n_std=1.5,
                           facecolor=col, alpha=0.15, edgecolor=col,
                           linewidth=1.3, linestyle='--', zorder=2)
        ax.scatter(cx, cy, color=col, s=60, zorder=4, edgecolors='white', lw=0.7)
        for i, name in enumerate(np.array(names)[mask]):
            ax.annotate(name, (cx[i], cy[i]), fontsize=6.5, ha='center', va='bottom',
                        xytext=(0, 4), textcoords='offset points', color=col)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xlabel('MDS Dim 1', fontsize=8)
    ax.set_ylabel('MDS Dim 2', fontsize=8)
    ax.grid(True, alpha=0.2, lw=0.4)
    ax.tick_params(labelsize=7)

def score_alignment(recipes, fs_products, labels_ing, labels_fs, threshold_ing):
    """Compute agreement % between ingredient and free sorting clusters."""
    common = [r for r in recipes if r in fs_products]
    fs_map  = {p: labels_fs[fs_products.index(p)] for p in common}
    ing_map = {r: labels_ing[recipes.index(r)] for r in common}
    fs_ids  = sorted(set(fs_map.values()))
    ing_ids = sorted(set(ing_map.values()))
    conf = np.zeros((len(fs_ids), len(ing_ids)), dtype=int)
    for r in common:
        conf[fs_ids.index(fs_map[r]), ing_ids.index(ing_map[r])] += 1
    row_ind, col_ind = linear_sum_assignment(-conf)
    fs_to_ing = {fs_ids[r]: ing_ids[c] for r, c in zip(row_ind, col_ind)}
    n_agree = sum(conf[r, c] for r, c in zip(row_ind, col_ind))
    pct = 100 * n_agree / len(common)
    return n_agree, len(common), pct, fs_to_ing, fs_map, ing_map, fs_ids, common

print('Helpers defined.')

## 5. OAV Cutoff Sweep — Find Best Cutoff

Test all cutoff levels and compare agreement with free sorting at matched cluster count.

In [ ]:
# First find the threshold that gives 4 clusters for each cutoff variant
cutoffs = [0.00, 0.01, 0.02, 0.05, 0.10, 0.15, 0.20]
results_sweep = []

for cutoff in cutoffs:
    _, vecs = build_cas_vectors(df, recipes, oav_cutoff_pct=cutoff)
    diss = cosine_dissimilarity(vecs)
    Z = linkage(squareform(diss, checks=False), method='ward')
    
    # Find threshold that gives 4 clusters
    t_4 = None
    for t in np.arange(0.02, 1.0, 0.02):
        lbl = fcluster(Z, t=t, criterion='distance')
        if len(np.unique(lbl)) == 4:
            t_4 = t
            break
    
    if t_4 is None:
        print(f'cutoff={cutoff:.0%}: cannot find 4-cluster threshold')
        continue
    
    labels_4 = fcluster(Z, t=t_4, criterion='distance')
    mds = MDS(n_components=2, dissimilarity='precomputed', metric=True,
              n_init=10, max_iter=1000, random_state=42, normalized_stress='auto')
    mds.fit(diss)
    
    n_agree, n_common, pct, _, _, _, _, _ = score_alignment(
        recipes, fs_products, labels_4, labels_fs, t_4)
    
    results_sweep.append({
        'cutoff': cutoff, 't_4clusters': t_4,
        'stress': mds.stress_,
        'agreement': pct, 'n_agree': n_agree,
    })
    print(f'cutoff={cutoff:.0%}  t={t_4:.2f}  stress={mds.stress_:.3f}  '
          f'agreement={n_agree}/{n_common} = {pct:.1f}%')

df_sweep = pd.DataFrame(results_sweep)
best_row = df_sweep.loc[df_sweep['agreement'].idxmax()]
print(f'\nBest: cutoff={best_row["cutoff"]:.0%}  agreement={best_row["agreement"]:.1f}%')

## 6. Best Model — Full Analysis

In [ ]:
BEST_CUTOFF = float(best_row['cutoff'])
BEST_T      = float(best_row['t_4clusters'])

print(f'Using cutoff={BEST_CUTOFF:.0%}, threshold={BEST_T}')

cas_vocab, vecs_best = build_cas_vectors(df, recipes, oav_cutoff_pct=BEST_CUTOFF)
diss_best = cosine_dissimilarity(vecs_best)
coords_best, Z_best, labels_best, stress_best = run_mds(diss_best, BEST_T)

n_agree, n_common, pct, fs_to_ing, fs_map, ing_map, fs_ids, common = score_alignment(
    recipes, fs_products, labels_best, labels_fs, BEST_T
)
ing_to_fs = {v: k for k, v in fs_to_ing.items()}

print(f'\nStress: {stress_best:.3f}')
print(f'Agreement: {n_agree}/{n_common} = {pct:.1f}%  (V18 baseline: 45.5%)')

In [ ]:
# Per-recipe detail
rows_out = []
for r in common:
    fs_c  = fs_map[r]
    ing_c = ing_map[r]
    agree = (fs_to_ing.get(fs_c) == ing_c)
    rows_out.append({'Recipe': r,
                     'FS Cluster': f'FS-C{fs_c}',
                     'CAS Cluster': f'CAS-C{ing_c}',
                     'Match': 'YES' if agree else 'NO'})

df_align = pd.DataFrame(rows_out).sort_values(['FS Cluster', 'Recipe'])
print(df_align.to_string(index=False))

print('\n--- Per Free Sorting Cluster ---')
for fs_c in fs_ids:
    sub = df_align[df_align['FS Cluster'] == f'FS-C{fs_c}']
    mm  = list(sub[sub['Match'] == 'NO']['Recipe'])
    print(f'  FS-C{fs_c} ({len(sub)} recipes): {(sub["Match"]=="YES").sum()}/{len(sub)} match'
          + (f'  mismatches: {mm}' if mm else '  ALL MATCH'))

## 7. Alignment Map — CAS Model vs Free Sorting

In [ ]:
fs_color_map = {c: CLUSTER_COLORS[i] for i, c in enumerate(fs_ids)}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Left: free sorting reference
mds_plot(ax1, xlstat_coords, xlstat_products, labels_fs_xlstat,
         'Free Sorting — reference (XLStat STATIS, 4 clusters)')
ax1.set_xlabel('Dim 1  (XLStat STATIS)', fontsize=9)
ax1.set_ylabel('Dim 2  (XLStat STATIS)', fontsize=9)

# Right: CAS MDS coloured by FS cluster (shows mismatches)
ax2.set_facecolor('#FAFAFA')
ax2.axhline(0, color='#CCCCCC', lw=0.7, zorder=1)
ax2.axvline(0, color='#CCCCCC', lw=0.7, zorder=1)

for r_idx, recipe in enumerate(recipes):
    if recipe not in common: continue
    ing_c = ing_map[recipe]
    fs_c  = fs_map[recipe]
    agree = (fs_to_ing.get(fs_c) == ing_c)
    x, y  = coords_best[r_idx]
    col   = fs_color_map[fs_c]
    ax2.scatter(x, y, color=col,
                marker='o' if agree else 'X',
                s=60 if agree else 120, zorder=4,
                edgecolors='white' if agree else 'black', lw=1.2)
    ax2.annotate(recipe, (x, y), fontsize=6.5, ha='center', va='bottom',
                 xytext=(0, 4), textcoords='offset points', color=col)

legend_els = [mpatches.Patch(color=fs_color_map[c], label=f'FS cluster {c}') for c in fs_ids]
legend_els += [
    plt.scatter([], [], marker='o', color='gray', s=50,  label='Agreement'),
    plt.scatter([], [], marker='X', color='gray', s=80,  label='Mismatch',
                edgecolors='black', lw=1.2),
]
ax2.legend(handles=legend_els, fontsize=8, loc='best', framealpha=0.9)
ax2.set_title(f'CAS-Level OAV Fingerprint (cutoff={BEST_CUTOFF:.0%})\nX = mismatch vs free sorting',
              fontsize=9, fontweight='bold')
ax2.set_xlabel('MDS Dim 1', fontsize=9)
ax2.set_ylabel('MDS Dim 2', fontsize=9)
ax2.grid(True, alpha=0.2, lw=0.4)
ax2.tick_params(labelsize=7)

fig.suptitle(
    f'V19: CAS-Level OAV Fingerprint vs Free Sorting\n'
    f'Agreement: {n_agree}/{n_common} = {pct:.1f}%  |  '
    f'stress={stress_best:.3f}  (V18 baseline: 45.5%, stress=0.687)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v19_cas_alignment.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

## 8. Cutoff Sweep Summary Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

cutoff_pct = [r['cutoff'] * 100 for r in results_sweep]
agreements = [r['agreement'] for r in results_sweep]
stresses   = [r['stress'] for r in results_sweep]

ax1.plot(cutoff_pct, agreements, 'o-', color='#E63946', lw=2)
ax1.axhline(45.5, color='gray', linestyle='--', lw=1, label='V18 baseline (45.5%)')
ax1.set_xlabel('OAV Cutoff (%)')
ax1.set_ylabel('Agreement with Free Sorting (%)')
ax1.set_title('Agreement vs OAV Cutoff', fontweight='bold')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 105)

ax2.plot(cutoff_pct, stresses, 'o-', color='#457B9D', lw=2)
ax2.axhline(0.687, color='gray', linestyle='--', lw=1, label='V18 stress (0.687)')
ax2.set_xlabel('OAV Cutoff (%)')
ax2.set_ylabel('MDS Stress')
ax2.set_title('MDS Stress vs OAV Cutoff', fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.suptitle('V19: OAV Cutoff Sweep (CAS-Level Fingerprint)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v19_cutoff_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sweep plot saved.')

## 9. What Do the Remaining Mismatches Have in Common?

Inspect the top OAV ingredients for mismatched recipes to understand *why* the model still disagrees with panelists.

In [ ]:
mismatches = [r for r in common if fs_to_ing.get(fs_map[r]) != ing_map[r]]
matches    = [r for r in common if fs_to_ing.get(fs_map[r]) == ing_map[r]]

print(f'Mismatched recipes ({len(mismatches)}): {mismatches}')
print()

for recipe in mismatches:
    rows_r = df[(df['Rez.-Nr.'] == recipe) & (df['Totalmenge'] > 0)].copy()
    rows_r = rows_r.sort_values('oav_proxy', ascending=False)
    top5 = rows_r[['Name', 'oav_proxy', 'thresh_ppm', 'Totalmenge']].head(5)
    fs_c  = fs_map[recipe]
    ing_c = ing_map[recipe]
    print(f'{recipe}  [FS=C{fs_c} → predicted C{ing_c}]  top OAV ingredients:')
    for _, row in top5.iterrows():
        oav_share = row['oav_proxy'] / rows_r['oav_proxy'].sum() * 100
        print(f'  {row["Name"][:45]:45s}  OAV%={oav_share:5.1f}%  '
              f'threshold={row["thresh_ppm"]}ppm')
    print()

## 10. Key Findings

### Method vs V18
| | V18 (OT1 category) | V19 (CAS fingerprint) |
|---|---|---|
| Feature dimensions | ~12 OT1 categories | 137 CAS numbers |
| Weighting | OAV proxy | OAV proxy |
| Sub-threshold filtering | No | Yes (OAV cutoff sweep) |
| Agreement with free sorting | 45.5% | TBD |
| MDS stress | 0.687 | TBD |

### What CAS-level fingerprint captures that OT1 categories miss
- Two recipes both labelled 'fruity' now look different if they use different fruity molecules
- Rare/unique ingredients that dominate OAV are properly represented
- FS-C4 failure (0% in V18) should improve if those 4 recipes share specific CAS numbers

### Remaining mismatches
If mismatches persist, the most likely explanations are:
1. **Psychophysical interactions** — ingredient combinations smell different from their sum (masking, enhancement)
2. **Missing threshold data** — 7.4% of ingredient mass has unknown thresholds, using fallback=1.0
3. **Concentration-dependent character** — some ingredients smell different at high vs low concentrations
4. **Panelist subjectivity** — free sorting with 4 panelists has noise; some groupings may not be chemically grounded